<a href="https://colab.research.google.com/github/kazi-amir/ml-learning/blob/main/Datathon-Practice/house-sale-price.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# House Sale Price Prediction

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sbn

In [2]:
train_df = pd.read_csv('train.csv')
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
train_df.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


In [4]:
train_df.dropna(subset=['SalePrice'], inplace=True)
train_df.shape

(1460, 81)

In [5]:
for col in train_df.columns:
  if train_df[col].isna().sum() > 1000:
    train_df.drop(col, axis=1, inplace=True)

train_df.shape

(1460, 77)

In [6]:
train_in = train_df.drop('SalePrice', axis=1)
train_out = train_df['SalePrice']

In [7]:
train_in.shape, train_out.shape

((1460, 76), (1460,))

In [8]:
numeric_cols = train_in.select_dtypes(include=np.number).columns
categorical_cols = train_in.select_dtypes(exclude=np.number).columns

In [9]:
if 'Id' in numeric_cols:
    numeric_cols = numeric_cols.drop('Id')

In [10]:
numeric_cols = numeric_cols.to_list()
categorical_cols = categorical_cols.to_list()

In [11]:
numeric_cols

['MSSubClass',
 'LotFrontage',
 'LotArea',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtFinSF1',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'MiscVal',
 'MoSold',
 'YrSold']

## Train Validation split

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
X_train, X_val, y_train, y_val = train_test_split(
    train_in,
    train_out,
    test_size=0.2,
    random_state=42
  )

In [14]:
X_train.shape, X_val.shape, y_train.shape, y_val.shape

((1168, 76), (292, 76), (1168,), (292,))

### Missing Values - Numeric

In [15]:
from sklearn.impute import SimpleImputer

In [16]:
imputer = SimpleImputer(strategy='mean')

In [17]:
imputer.fit(X_train[numeric_cols])

SimpleImputer()

In [18]:
X_train[numeric_cols] = imputer.transform(X_train[numeric_cols])
X_val[numeric_cols] = imputer.transform(X_val[numeric_cols])

In [19]:
X_train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
254,255,20.0,RL,70.0,8400.0,Pave,Reg,Lvl,AllPub,Inside,...,0.0,0.0,0.0,0.0,0.0,0.0,6.0,2010.0,WD,Normal
1066,1067,60.0,RL,59.0,7837.0,Pave,IR1,Lvl,AllPub,Inside,...,40.0,0.0,0.0,0.0,0.0,0.0,5.0,2009.0,WD,Normal
638,639,30.0,RL,67.0,8777.0,Pave,Reg,Lvl,AllPub,Inside,...,0.0,164.0,0.0,0.0,0.0,0.0,5.0,2008.0,WD,Normal
799,800,50.0,RL,60.0,7200.0,Pave,Reg,Lvl,AllPub,Corner,...,0.0,264.0,0.0,0.0,0.0,0.0,6.0,2007.0,WD,Normal
380,381,50.0,RL,50.0,5000.0,Pave,Reg,Lvl,AllPub,Inside,...,0.0,242.0,0.0,0.0,0.0,0.0,5.0,2010.0,WD,Normal


### Feature Scaling - MinMaxScaler

In [20]:
from sklearn.preprocessing import MinMaxScaler

In [21]:
scaler = MinMaxScaler()

In [22]:
scaler.fit(X_train[numeric_cols])

MinMaxScaler()

In [23]:
X_train[numeric_cols] = scaler.transform(X_train[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])

In [24]:
X_train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
254,255,0.000000,RL,0.167808,0.033186,Pave,Reg,Lvl,AllPub,Inside,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.454545,1.00,WD,Normal
1066,1067,0.235294,RL,0.130137,0.030555,Pave,IR1,Lvl,AllPub,Inside,...,0.073126,0.000000,0.0,0.0,0.0,0.0,0.363636,0.75,WD,Normal
638,639,0.058824,RL,0.157534,0.034948,Pave,Reg,Lvl,AllPub,Inside,...,0.000000,0.297101,0.0,0.0,0.0,0.0,0.363636,0.50,WD,Normal
799,800,0.176471,RL,0.133562,0.027577,Pave,Reg,Lvl,AllPub,Corner,...,0.000000,0.478261,0.0,0.0,0.0,0.0,0.454545,0.25,WD,Normal
380,381,0.176471,RL,0.099315,0.017294,Pave,Reg,Lvl,AllPub,Inside,...,0.000000,0.438406,0.0,0.0,0.0,0.0,0.363636,1.00,WD,Normal


### Missing Values - Categoric

In [25]:
X_train[categorical_cols] = X_train[categorical_cols].fillna('Unknown')
X_val[categorical_cols] = X_val[categorical_cols].fillna('Unknown')

### One Hot Encodig -Categoric

In [26]:
from sklearn.preprocessing import OneHotEncoder

In [27]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [28]:
encoder.fit(X_train[categorical_cols])

OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [29]:
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))

In [30]:
train_encoded = pd.DataFrame(
    encoder.transform(X_train[categorical_cols]),
    columns=encoded_cols,
    index=X_train.index
)

val_encoded = pd.DataFrame(
    encoder.transform(X_val[categorical_cols]),
    columns=encoded_cols,
    index=X_val.index
)

In [31]:
X_train = pd.concat([X_train, train_encoded], axis=1)
X_val = pd.concat([X_val, val_encoded], axis=1)

In [32]:
X_train.shape, X_val.shape

((1168, 324), (292, 324))

In [33]:
for col in X_train.columns:
  if X_train[col].isna().sum() > 0:
    print(col, X_train[col].dtype)

In [34]:
X_train = X_train[numeric_cols + encoded_cols]
X_val = X_val[numeric_cols + encoded_cols]

In [35]:
X_train.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
254,0.000000,0.167808,0.033186,0.444444,0.625,0.615942,0.116667,0.000000,0.163359,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1066,0.235294,0.130137,0.030555,0.555556,0.750,0.876812,0.733333,0.000000,0.000000,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
638,0.058824,0.157534,0.034948,0.444444,0.750,0.275362,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
799,0.176471,0.133562,0.027577,0.444444,0.750,0.471014,0.000000,0.182874,0.100815,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
380,0.176471,0.099315,0.017294,0.444444,0.625,0.376812,0.000000,0.000000,0.038625,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [36]:
X_train.shape, X_val.shape, y_train.shape, y_val.shape

((1168, 284), (292, 284), (1168,), (292,))

## Global Result Array

In [37]:
results = []

## Model Evaluation

In [38]:
def rmse(targets, predictions):
    t_vals = np.asarray(targets).flatten()
    p_vals = np.asarray(predictions).flatten()

    return np.sqrt(np.mean(np.square(t_vals - p_vals)))

In [39]:
def mae(targets, predictions):
    t_vals = np.asarray(targets).flatten()
    p_vals = np.asarray(predictions).flatten()

    return np.mean(np.abs(t_vals - p_vals))

In [40]:
def evaluate(model, targets, predictions):
  rmse_val = rmse(targets, predictions)
  mae_val = mae(targets, predictions)

  results.append({
      "model": model,
      "rmse": rmse_val,
      "mae": mae_val
    })

  print(f"RMSE: {rmse_val}, MAE: {mae_val}")

## Model Building

### Linear Regression

In [41]:
from sklearn.linear_model import LinearRegression

In [42]:
linear_model = LinearRegression()

In [43]:
linear_model.fit(X_train, y_train)

LinearRegression()

In [44]:
linear_preds = linear_model.predict(X_val)

In [45]:
evaluate("Linear Regression", y_val, linear_preds)

RMSE: 29604.187370329455, MAE: 18597.01571703162


### Decision Tree Regressor

In [46]:
from sklearn.tree import DecisionTreeRegressor

In [47]:
dt_model = DecisionTreeRegressor(random_state=42)

In [48]:
dt_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [49]:
dt_preds = dt_model.predict(X_val)

In [50]:
evaluate("Decision Tree Regressor", y_val, dt_preds)

RMSE: 42137.391174422526, MAE: 27972.678082191782


### Decision Tree Regresson - max_depth (7)

In [51]:
dt_model_mxdpth = DecisionTreeRegressor(max_depth=7, random_state=42)

In [52]:
dt_model_mxdpth.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=7, random_state=42)

In [53]:
dt_mxdpth_preds = dt_model_mxdpth.predict(X_val)

In [54]:
evaluate("Decision Tree Regressor - max_depth (7)", y_val, dt_mxdpth_preds)

RMSE: 37545.714150629196, MAE: 25337.302126027447


### Random Forrest Regressor

In [55]:
from sklearn.ensemble import RandomForestRegressor

In [56]:
rf_model = RandomForestRegressor(random_state=42)

In [57]:
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [58]:
rf_preds = rf_model.predict(X_val)

In [59]:
evaluate("Random Forrest Regressor", y_val, rf_preds)

RMSE: 28473.158992951765, MAE: 17587.307979452053


### XGBoost

In [60]:
from xgboost import XGBRegressor

In [61]:
xgb_model = XGBRegressor(random_state=42)

In [62]:
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [63]:
xgb_preds = xgb_model.predict(X_val)

In [64]:
evaluate("XGBoost", y_val, xgb_preds)

RMSE: 28163.061409150814, MAE: 17683.912778253423


### XGBRegression - n_estimator = 1000

In [ ]:
from xgboost import XGBRegressor

In [66]:
xgb1000_model = XGBRegressor(random_state=42, n_estimator = 1000)

In [67]:
xgb1000_model.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [20:45:39] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimator=1000,
             n_estimators=None, n_jobs=None, ...)

In [68]:
xgb_preds = xgb1000_model.predict(X_val)

In [69]:
evaluate("XGBoost", y_val, xgb_preds)

RMSE: 28163.061409150814, MAE: 17683.912778253423


## Test Data Preprocessing

In [ ]:
test_df = pd.read_csv('test.csv')
test_df.shape

In [ ]:
test_ids = test_df['Id']

In [ ]:
test_in = test_df[train_in.columns]
test_in.shape

In [ ]:
test_in[numeric_cols] = imputer.transform(test_in[numeric_cols])

In [ ]:
test_in[numeric_cols] = scaler.transform(test_in[numeric_cols])

In [ ]:
test_in[categorical_cols] = test_in[categorical_cols].fillna('Unknown')

In [ ]:
test_encoded = pd.DataFrame(
    encoder.transform(test_in[categorical_cols]),
    columns=encoded_cols,
    index=test_in.index
)

In [ ]:
test_in = pd.concat([test_in, test_encoded], axis=1)

In [ ]:
X_test = test_in[numeric_cols + encoded_cols]
X_test.shape

## Kaggle Submission

In [ ]:
test_preds = xgb_model.predict(X_test)

In [ ]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_preds
})
submission.head()

In [ ]:
submission.to_csv('submission.csv', index=False)